---
title: "Machine Learning: Learning with Limited, Weak, and Noisy Supervision"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---


<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/15-limited-weak-noisy-supervision.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Learning with Limited, Weak, and Noisy Supervision**

Supervised learning is usually introduced as if a dataset arrived with a correct target beside every observation. In practice, labels are produced by people, instruments, business processes, heuristic rules, or older models. They have a **provenance**, a **cost**, a **delay**, and an **error mechanism**. Learning under imperfect supervision asks how to extract useful signal without pretending that all available targets are equivalent to independent expert judgments.

The central distinction is between the observation $x$, the latent target of interest $y$, and the supervision actually recorded by the pipeline. Depending on the setting, that recorded signal may be absent, selected, aggregated, delayed, or corrupted. Methods in this chapter modify one or more of four resources:

- the **data distribution**, by exploiting structure in unlabeled observations;
- the **label-acquisition policy**, by deciding which observations deserve expert attention;
- the **supervision model**, by representing how weak sources or noise generate observed labels;
- the **training objective and decision policy**, by making learning robust to unreliable or imbalanced targets.

These methods can be combined, but they are not interchangeable. Self-training assumes that confident predictions are often correct. Active learning assumes that an annotator can answer selected queries. Weak supervision assumes that noisy sources contain complementary information. Noise correction assumes something identifiable about the corruption process. A method can amplify error when its assumption is false.

### **Why Supervision Becomes the Bottleneck**

Labels become a bottleneck for several different reasons:

| Supervision problem | What is observed | What is missing or unreliable | Natural starting point |
|---|---|---|---|
| Few clean labels | A small trusted labeled set and many unlabeled examples | Coverage of the input distribution | Semi-supervised or self-supervised learning |
| Expensive expert labels | An unlabeled pool and a limited annotation budget | Which examples are worth labeling | Active learning |
| Programmatic or distant labels | Rules, database matches, crowd votes, or model outputs | Source reliability and dependence | Weak supervision and label models |
| Positive-unlabeled data | Selected positives and an unlabeled mixture | Explicit negative labels and selection propensity | PU learning |
| Corrupted labels | A label for most examples | The clean label and the noise process | Robust losses, noise models, auditing |
| Rare events | Mostly correct labels but very low positive prevalence | Enough minority evidence and an appropriate action threshold | Cost sensitivity, resampling, thresholding |
| Subjective targets | Multiple defensible judgments | A unique ground truth | Annotator modeling and distributional targets |

<div class="diagram-scroll">

![A map from supervision bottlenecks to appropriate learning strategies.](assets/supervision-bottleneck-map.svg){fig-alt="Limited labels, unlabeled pools, weak labels, noisy labels, and rare events are mapped to semi-supervised, self-supervised, active, weak-supervision, and robust-learning strategies."}

</div>

Label cost is not just the price per click. A realistic budget can include discovery, adjudication, and rework:

$$
C_{\text{total}}
=n_{\text{query}}c_{\text{annotation}}
+n_{\text{disputed}}c_{\text{adjudication}}
+c_{\text{guideline}}
+c_{\text{quality control}}
+c_{\text{delay}}.
$$

The value of another label depends on what it covers and how reliably it is produced. Ten independent labels from a missing subgroup may be more valuable than ten thousand duplicate labels from an easy region. Disagreement also has more than one meaning: it may reveal careless work, an unclear guideline, genuine ambiguity, or a target that should be represented as a distribution rather than a forced class.

<details>
<summary><strong>Python: separate annotator agreement from majority-vote accuracy</strong></summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score

# The latent labels are available only because this is a controlled simulation.
y_true = np.array([0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0])
annotator_a = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0])
annotator_b = np.array([0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0])
annotator_c = np.array([0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0])

votes = np.vstack([annotator_a, annotator_b, annotator_c])
majority = (votes.sum(axis=0) >= 2).astype(int)
vote_probability = votes.mean(axis=0)

# Entropy is high near a 50/50 vote and zero under unanimous agreement.
eps = 1e-12
entropy = -(
    vote_probability * np.log2(vote_probability + eps)
    + (1 - vote_probability) * np.log2(1 - vote_probability + eps)
)

pairwise_kappa = [
    cohen_kappa_score(votes[i], votes[j])
    for i in range(3)
    for j in range(i + 1, 3)
]

print("majority-vote accuracy:", round(accuracy_score(y_true, majority), 3))
print("mean pairwise Cohen's kappa:", round(float(np.mean(pairwise_kappa)), 3))
print("items with disagreement:", np.flatnonzero(entropy > 0).tolist())
print("soft targets:", np.round(vote_probability, 2).tolist())
```

</details>

A majority label is useful for training, but it erases the difference between a unanimous $3$-to-$0$ decision and a disputed $2$-to-$1$ decision. Store raw annotations, annotator identities when ethically and legally appropriate, timestamps, guidelines, and adjudication outcomes. The history is often more valuable than the final integer label.


### **Semi-Supervised Learning**

Semi-supervised learning uses a small labeled set $L=\{(x_i,y_i)\}$ together with a larger unlabeled set $U=\{x_j\}$. Unlabeled observations reveal the marginal distribution $p(x)$, but the task requires $p(y\mid x)$. Consequently, $U$ helps only through an assumption that connects the geometry of $p(x)$ to the target boundary.

#### **Cluster, Smoothness, and Manifold Assumptions**

Three related assumptions explain most classical semi-supervised methods:

- **Smoothness assumption:** sufficiently similar observations should have similar conditional label distributions.
- **Cluster or low-density separation assumption:** high-density regions tend to share a label, so a useful decision boundary should pass through low-density space.
- **Manifold assumption:** high-dimensional observations concentrate near a lower-dimensional structure, and labels vary smoothly along that structure rather than through the ambient coordinate system.

<div class="diagram-scroll">

![Cluster, smoothness, and manifold assumptions for semi-supervised learning.](assets/semi-supervised-assumptions.svg){fig-alt="Three panels show a boundary through a low-density gap, smoothly changing class probability along nearby points, and samples on a curved lower-dimensional manifold."}

</div>

The assumptions are inductive biases, not facts supplied by unlabeled data. Consider two classes separated by color but occupying the same geometric cluster. Adding unlabeled points makes the cluster estimate more precise without revealing color labels. Similarly, a narrow bridge of ambiguous observations may connect otherwise separate clusters and cause graph propagation to spread a wrong label. A reliable experiment must therefore compare against a supervised baseline at the same clean-label budget and test several labeled-set seeds.

#### **Self-Training and Pseudo-Labels**

**Self-training** turns a probabilistic supervised estimator into an iterative teacher. At iteration $t$:

1. fit the model on the currently labeled set $L_t$;
2. predict class probabilities for $U_t$;
3. select examples whose confidence or rank passes a rule;
4. assign hard or soft pseudo-labels;
5. move selected examples into the training set and repeat.

For a hard pseudo-label $\hat y_j=\arg\max_c p_{\theta_t}(c\mid x_j)$ and confidence $q_j=\max_c p_{\theta_t}(c\mid x_j)$, a typical objective is

$$
\mathcal L(\theta)
=\frac{1}{|L|}\sum_{(x_i,y_i)\in L}\ell(y_i,p_\theta(x_i))
+\lambda_t\frac{1}{|P_t|}\sum_{j\in P_t}w(q_j)\ell(\hat y_j,p_\theta(x_j)),
$$

where $P_t=\{j:q_j\ge\tau_t\}$ is the accepted pseudo-label set. The weight $\lambda_t$ or threshold $\tau_t$ is often scheduled so uncertain synthetic targets do not dominate early training.

<div class="diagram-scroll">

![The self-training loop and its confirmation-bias failure path.](assets/self-training-loop.svg){fig-alt="A small clean seed trains a teacher, which predicts an unlabeled pool; confident pseudo-labels train a student and the loop repeats, while confident mistakes can feed back as confirmation bias."}

</div>

Confidence is not the same as correctness. A model can be confidently wrong under distribution shift, class imbalance, poor calibration, or a missing class. Useful safeguards include probability calibration on clean validation data, class-aware thresholds, soft targets, strong augmentation, an exponential-moving-average teacher, caps on the pseudo-label ratio, and manual auditing of accepted examples. The threshold is a model-selection parameter and must never be tuned against the test set.

<details>
<summary><strong>Python: compare a label-only baseline with self-training</strong></summary>

```python
import numpy as np
from sklearn.base import clone
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

X, y = make_moons(n_samples=1200, noise=0.22, random_state=15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=15
)

rng = np.random.default_rng(15)
labeled_index = np.concatenate([
    rng.choice(np.flatnonzero(y_train == class_id), size=12, replace=False)
    for class_id in np.unique(y_train)
])
y_semi = np.full_like(y_train, fill_value=-1)
y_semi[labeled_index] = y_train[labeled_index]

base = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=2.0, max_iter=2000),
)

supervised = clone(base).fit(X_train[labeled_index], y_train[labeled_index])
self_training = SelfTrainingClassifier(
    estimator=clone(base), threshold=0.88, max_iter=20
).fit(X_train, y_semi)

selected = self_training.labeled_iter_ > 0
pseudo_label_precision = accuracy_score(
    y_train[selected], self_training.transduction_[selected]
) if selected.any() else float("nan")

print("clean labels:", len(labeled_index))
print("pseudo-labels accepted:", int(selected.sum()))
print("pseudo-label precision (simulation audit):", round(pseudo_label_precision, 3))
print("label-only test accuracy:", round(accuracy_score(y_test, supervised.predict(X_test)), 3))
print("self-training test accuracy:", round(accuracy_score(y_test, self_training.predict(X_test)), 3))
```

</details>

The hidden training labels are inspected only to audit the simulation. A real pipeline would estimate pseudo-label quality from a separate clean audit sample. One run is not evidence that self-training helps: repeat the comparison across labeled subsets, especially because the first few seed labels determine which errors can be reinforced.


#### **Graph-Based Methods**

Graph-based semi-supervised learning treats every labeled and unlabeled observation as a node. An edge weight $W_{ij}\ge 0$ measures similarity, often through a $k$-nearest-neighbour graph or an RBF kernel. The prediction vector $f_i$ should vary slowly across strong edges while respecting observed labels.

A representative objective is

$$
\min_F\quad
\frac{1}{2}\sum_{i,j}W_{ij}\lVert F_i-F_j\rVert_2^2
+\mu\sum_{i\in L}\lVert F_i-Y_i\rVert_2^2.
$$

The first term is graph smoothness. With degree matrix $D_{ii}=\sum_j W_{ij}$ and graph Laplacian $L_G=D-W$, it can be written as $\operatorname{tr}(F^\top L_G F)$. The second term anchors predictions to observed labels. **Label propagation** clamps labeled nodes strongly; **label spreading** uses normalized graph geometry and soft clamping, allowing some resistance to mislabeled seeds.

<div class="diagram-scroll">

![Labels spreading over a weighted similarity graph.](assets/graph-label-spreading.svg){fig-alt="Blue and red labeled nodes connect through weighted edges to unlabeled nodes; a smoothness objective and label anchors produce soft class probabilities."}

</div>

The graph is the model. Poor feature scaling, an inappropriate metric, disconnected components without labels, hubs, or edges that cross semantic classes all change the answer. A dense RBF graph costs roughly $O(n^2)$ memory; large systems often need sparse nearest-neighbour graphs, approximate search, subsampling, or inductive alternatives.

<details>
<summary><strong>Python: propagate a few labels over a nearest-neighbour graph</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelSpreading
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=1000, noise=0.18, random_state=21)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=21
)

rng = np.random.default_rng(21)
labeled_index = np.concatenate([
    rng.choice(np.flatnonzero(y_train == class_id), 10, replace=False)
    for class_id in [0, 1]
])
y_graph = np.full_like(y_train, -1)
y_graph[labeled_index] = y_train[labeled_index]

# Scaling matters because graph edges are built from distances.
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(max_iter=2000).fit(
    X_train_scaled[labeled_index], y_train[labeled_index]
)
spreading = LabelSpreading(
    kernel="knn", n_neighbors=12, alpha=0.20, max_iter=100
).fit(X_train_scaled, y_graph)

print("label-only accuracy:", round(accuracy_score(y_test, baseline.predict(X_test_scaled)), 3))
print("label-spreading accuracy:", round(accuracy_score(y_test, spreading.predict(X_test_scaled)), 3))
print("iterations to convergence:", spreading.n_iter_)
```

</details>

#### **Consistency Regularization**

Consistency regularization asks a model to produce compatible predictions for different label-preserving views of the same observation. If $a_w$ is a weak transformation and $a_s$ is a stronger transformation, one common unlabeled loss is

$$
\mathcal L_u
=\mathbb E_{x\in U}
\left[
\mathbf 1\!\left(\max q\ge\tau\right)
H\!\left(\operatorname{stopgrad}(q),p_\theta(\cdot\mid a_s(x))\right)
\right],
\qquad
q=p_{\bar\theta}(\cdot\mid a_w(x)).
$$

The teacher parameters $\bar\theta$ may be the current model or an exponential-moving average. The indicator applies the loss only when the teacher is sufficiently confident. `stopgrad` prevents the target distribution from moving merely to reduce the same loss. For regression, the discrepancy may be squared error; for classification it is often cross-entropy or KL divergence.

<div class="diagram-scroll">

![Consistency regularization with weak and strong views.](assets/consistency-regularization.svg){fig-alt="An unlabeled input produces a weak teacher view and a strong student view; a confidence-gated loss makes predictions agree when both transformations preserve the label."}

</div>

The difficult part is defining a valid invariance. Small image translations may preserve object identity; arbitrary text deletion may reverse sentiment; time-series warping may destroy an event's duration. Consistency is harmful when the augmentation changes the true target. It may also collapse to trivial predictions unless combined with supervised anchors, confidence control, balanced pseudo-labels, or contrastive structure.

<details>
<summary><strong>Python: use agreement across perturbations as a pseudo-label gate</strong></summary>

```python
import numpy as np
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=1600, n_features=12, n_informative=7,
    class_sep=1.15, flip_y=0.02, random_state=22
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=22
)
rng = np.random.default_rng(22)
labeled = np.concatenate([
    rng.choice(np.flatnonzero(y_train == c), 20, replace=False) for c in [0, 1]
])
unlabeled = np.setdiff1d(np.arange(len(X_train)), labeled)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
teacher = clone(model).fit(X_train[labeled], y_train[labeled])

# Two feature perturbations approximate two valid measurement views.
scale = X_train.std(axis=0, ddof=1)
view_a = X_train[unlabeled] + rng.normal(0, 0.04 * scale, size=(len(unlabeled), X.shape[1]))
view_b = X_train[unlabeled] + rng.normal(0, 0.09 * scale, size=(len(unlabeled), X.shape[1]))
proba_a = teacher.predict_proba(view_a)
proba_b = teacher.predict_proba(view_b)

label_a = proba_a.argmax(axis=1)
label_b = proba_b.argmax(axis=1)
confidence = np.minimum(proba_a.max(axis=1), proba_b.max(axis=1))
accept = (label_a == label_b) & (confidence >= 0.90)

accepted_index = unlabeled[accept]
student_X = np.vstack([X_train[labeled], X_train[accepted_index]])
student_y = np.concatenate([y_train[labeled], label_a[accept]])
student = clone(model).fit(student_X, student_y)

print("accepted pseudo-labels:", int(accept.sum()))
print("accepted-label precision (simulation):", round(accuracy_score(y_train[accepted_index], label_a[accept]), 3))
print("teacher test accuracy:", round(accuracy_score(y_test, teacher.predict(X_test)), 3))
print("student test accuracy:", round(accuracy_score(y_test, student.predict(X_test)), 3))
```

</details>

This compact example uses perturbation agreement as a selection rule rather than differentiating a neural consistency loss. It exposes the essential controls: two valid views, confidence, agreement, and a clean evaluation set. In neural methods, the strong-view loss is optimized directly and usually combined with augmentation schedules and teacher updates.


### **Self-Supervised Learning**

Self-supervised learning creates supervisory targets from structure already present in the observations. No human class label is needed during pretraining, but the learning signal is not assumption-free: the chosen prediction task and transformations specify what information the representation should retain or ignore. The learned encoder is then evaluated on downstream tasks through a linear probe, fine-tuning, retrieval, or few-label learning curve.

#### **Pretext and Predictive Objectives**

A **pretext task** is an automatically generated task used to teach a representation. Common families include:

- **masked prediction:** hide words, image patches, or sensor channels and predict the missing content;
- **autoregressive prediction:** predict future tokens, frames, or time steps from context;
- **denoising and reconstruction:** corrupt an input and recover the clean version;
- **transformation prediction:** predict rotation, temporal order, jigsaw position, or another generated transformation;
- **cross-modal prediction:** predict one view or modality from another, such as text paired with an image.

For a corruption process $\tilde x\sim q(\tilde x\mid x)$ and generated target $t(x,\tilde x)$, pretraining solves

$$
\min_{\theta,\phi}
\mathbb E_{x,\tilde x}
\left[\ell\!\left(t(x,\tilde x),g_\phi(f_\theta(\tilde x))\right)\right],
$$

where $f_\theta$ is the encoder and $g_\phi$ is a task-specific prediction head. After pretraining, the head may be discarded while the encoder is reused. The pretext target should require semantics useful to the downstream problem. If a rotation classifier succeeds by detecting interpolation artifacts, its accuracy is high while its representation may be poor.

<div class="diagram-scroll">

![Self-supervised pretraining from automatically derived targets.](assets/self-supervised-objectives.svg){fig-alt="Raw unlabeled data is converted into masked, predictive, denoising, transformation, or contrastive targets, used to pretrain an encoder, then evaluated on downstream transfer."}

</div>

<details>
<summary><strong>Python: construct and solve a rotation-prediction pretext task</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

images = load_digits().images
image_ids = np.arange(len(images))
train_ids, test_ids = train_test_split(image_ids, test_size=0.30, random_state=15)

def rotation_dataset(source_images):
    transformed, targets = [], []
    for rotation in range(4):
        # k=0,1,2,3 corresponds to 0, 90, 180, and 270 degrees.
        rotated = np.rot90(source_images, k=rotation, axes=(1, 2))
        transformed.append(rotated.reshape(len(rotated), -1))
        targets.append(np.full(len(rotated), rotation))
    return np.vstack(transformed), np.concatenate(targets)

# Split original images before augmentation so rotated siblings never cross the split.
X_pretrain, y_pretrain = rotation_dataset(images[train_ids])
X_pretest, y_pretest = rotation_dataset(images[test_ids])

rotation_model = LogisticRegression(max_iter=2500, C=1.0).fit(X_pretrain, y_pretrain)
pretext_accuracy = accuracy_score(y_pretest, rotation_model.predict(X_pretest))

print("pretext train examples:", len(X_pretrain))
print("held-out rotation accuracy:", round(pretext_accuracy, 3))
```

</details>

This demonstrates automatic target construction and leakage-safe splitting. A complete representation-learning experiment would expose an encoder layer, freeze or fine-tune it on a separate labeled task, and compare it with random initialization at several label budgets. Pretext accuracy alone is not the final criterion.

#### **Contrastive Learning**

Contrastive learning defines two transformed views of the same observation as a **positive pair** and other observations as **negatives**. An encoder and projection head map views to normalized vectors $z$. A common InfoNCE loss for anchor $i$ and its positive partner $j(i)$ is

$$
\ell_i
=-\log
\frac{\exp(\operatorname{sim}(z_i,z_{j(i)})/\tau)}
{\sum_{k\ne i}\exp(\operatorname{sim}(z_i,z_k)/\tau)},
$$

where $\operatorname{sim}$ is usually cosine similarity and $\tau>0$ is the temperature. The numerator pulls positive views together; the denominator pushes the anchor away from competing views. A small $\tau$ concentrates the softmax on the hardest competitors and produces sharper gradients.

The method teaches **invariance to augmentation** and **discrimination between instances**. This makes augmentation design central. Cropping two views so they no longer contain the same object creates a false positive. Treating two different examples from the same semantic class as negatives creates false negatives. Large batches, memory banks, momentum encoders, clustering, or negative-free objectives address different parts of the sampling and collapse problem.

<details>
<summary><strong>Python: compute a symmetric InfoNCE loss</strong></summary>

```python
import numpy as np
from scipy.special import logsumexp

rng = np.random.default_rng(150)
n_samples, dimension = 64, 32
latent = rng.normal(size=(n_samples, dimension))

# Two noisy views preserve the same latent instance identity.
view_a = latent + rng.normal(scale=0.15, size=latent.shape)
view_b = latent + rng.normal(scale=0.15, size=latent.shape)

def normalize(rows):
    return rows / np.linalg.norm(rows, axis=1, keepdims=True)

def symmetric_info_nce(first, second, temperature=0.2):
    z = np.vstack([normalize(first), normalize(second)])
    logits = z @ z.T / temperature
    np.fill_diagonal(logits, -np.inf)  # an embedding is not its own negative

    n = len(first)
    positive_index = np.concatenate([np.arange(n, 2 * n), np.arange(n)])
    positive_logits = logits[np.arange(2 * n), positive_index]
    return float(np.mean(-positive_logits + logsumexp(logits, axis=1)))

aligned_loss = symmetric_info_nce(view_a, view_b)
shuffled_loss = symmetric_info_nce(view_a, view_b[rng.permutation(n_samples)])

print("loss with correct positive pairs:", round(aligned_loss, 3))
print("loss with shuffled positive pairs:", round(shuffled_loss, 3))
```

</details>

The aligned loss should be much lower because corresponding views share instance-level signal. In an actual model, gradients update the encoder so that this relationship emerges rather than being supplied by a synthetic latent vector. Representation quality should still be measured by transfer, robustness, and subgroup behavior, not by contrastive loss alone.


### **Active Learning**

Active learning changes data collection from a passive sample into a sequential decision. Given a labeled set $L_t$, an unlabeled pool $U_t$, a model, and an annotation budget, a query policy chooses a batch $Q_t\subset U_t$ whose labels are expected to improve the system most. The cycle is

$$
L_t\;\longrightarrow\;\text{fit model}\;\longrightarrow\;\text{score }U_t
\;\longrightarrow\;\text{query humans}\;\longrightarrow\;L_{t+1}.
$$

Active learning can reduce annotation work when the pool resembles deployment data, the model's uncertainty is informative, selected items can actually be labeled, and the retraining loop is fast enough. It can fail by repeatedly selecting outliers, ambiguous items, duplicates, or examples from one dense subgroup.

<div class="diagram-scroll">

![A pool-based active-learning and annotation loop.](assets/active-learning-loop.svg){fig-alt="A clean labeled seed trains a model, which scores an unlabeled pool for uncertainty, diversity, and cost; selected examples are annotated, audited, and added before the next iteration."}

</div>

#### **Uncertainty, Diversity, and Query Strategies**

For multiclass probabilities $p_c(x)$, standard uncertainty scores are:

$$
\begin{aligned}
u_{\text{least}}(x)&=1-\max_c p_c(x),\\
u_{\text{margin}}(x)&=1-\left(p_{(1)}(x)-p_{(2)}(x)\right),\\
u_{\text{entropy}}(x)&=-\sum_c p_c(x)\log p_c(x),
\end{aligned}
$$

where $p_{(1)}\ge p_{(2)}$ are the two largest probabilities. Least confidence asks whether the winning class is weak; margin sampling asks whether the top two classes are close; entropy measures uncertainty over the full distribution. These scores inherit model miscalibration and may be overconfident far from training data.

Uncertainty alone can select a batch of near duplicates. **Diversity-aware** policies combine informativeness with coverage, for example by clustering candidate embeddings, using a farthest-first or $k$-center rule, applying determinantal point processes, or optimizing a submodular objective. Other policies use disagreement among models (**query by committee**), expected model change, expected error reduction, density weighting, or explicit annotation cost.

<details>
<summary><strong>Python: compare uncertainty sampling with random acquisition</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=2400, n_features=16, n_informative=8, n_redundant=3,
    class_sep=1.0, flip_y=0.03, random_state=151
)
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=151
)

def run_active_learning(strategy, seed, rounds=8, batch_size=20):
    rng = np.random.default_rng(seed)
    labeled = np.concatenate([
        rng.choice(np.flatnonzero(y_pool == class_id), 5, replace=False)
        for class_id in [0, 1]
    ])
    available = np.setdiff1d(np.arange(len(X_pool)), labeled)
    history = []

    for round_id in range(rounds + 1):
        model = make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000)
        ).fit(X_pool[labeled], y_pool[labeled])
        history.append((len(labeled), accuracy_score(y_test, model.predict(X_test))))
        if round_id == rounds:
            break

        if strategy == "uncertainty":
            probabilities = model.predict_proba(X_pool[available])
            entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
            chosen_local = np.argsort(entropy)[-batch_size:]
        else:
            chosen_local = rng.choice(len(available), batch_size, replace=False)

        chosen = available[chosen_local]
        labeled = np.concatenate([labeled, chosen])
        available = np.setdiff1d(available, chosen, assume_unique=False)

    return history

records = []
for seed in range(8):
    for strategy in ["random", "uncertainty"]:
        for budget, accuracy in run_active_learning(strategy, 1000 + seed):
            records.append({"strategy": strategy, "labels": budget, "accuracy": accuracy})

summary = (
    pd.DataFrame(records)
    .groupby(["strategy", "labels"])["accuracy"]
    .agg(["mean", "std"])
    .round(3)
)
print(summary)
```

</details>

The random baseline is indispensable. Active policies have high variance with tiny seeds, and an uncertainty strategy that wins for one pool order may lose for another. The horizontal axis should ideally be annotation time or money rather than query count.

<details>
<summary><strong>Python: add diversity to an uncertain candidate batch</strong></summary>

```python
import numpy as np
from sklearn.metrics import pairwise_distances

def uncertainty_diversity_batch(X_pool, uncertainty, batch_size, candidate_factor=8):
    """Choose diverse points from the most uncertain candidate subset."""
    candidate_count = min(len(X_pool), candidate_factor * batch_size)
    candidate_id = np.argsort(uncertainty)[-candidate_count:]
    candidates = X_pool[candidate_id]

    # Begin with the most uncertain point.
    selected_local = [int(np.argmax(uncertainty[candidate_id]))]
    distance = pairwise_distances(candidates, candidates[selected_local]).ravel()

    # Farthest-first traversal discourages near-duplicate queries.
    while len(selected_local) < batch_size:
        next_local = int(np.argmax(distance))
        selected_local.append(next_local)
        distance = np.minimum(
            distance,
            pairwise_distances(candidates, candidates[[next_local]]).ravel(),
        )
    return candidate_id[np.array(selected_local)]

rng = np.random.default_rng(152)
candidate_X = rng.normal(size=(500, 6))
candidate_uncertainty = rng.uniform(size=500)
batch = uncertainty_diversity_batch(candidate_X, candidate_uncertainty, batch_size=12)
print("selected pool indexes:", batch.tolist())
print("mean selected uncertainty:", round(float(candidate_uncertainty[batch].mean()), 3))
```

</details>

This heuristic first restricts attention to uncertain candidates, then maximizes feature-space coverage. In production, distances should be computed in a validated representation, feature scales should be controlled, and diversity may need subgroup or geographic constraints rather than generic Euclidean spacing.

#### **Human-in-the-Loop Annotation**

The annotator is part of the learning system, not an infallible oracle. A robust loop specifies:

- a task definition with positive, negative, edge, and abstention examples;
- what context the annotator may inspect and what information must remain hidden;
- duplicate and gold-standard checks without turning quality control into surveillance;
- an adjudication route for ambiguity and disagreement;
- batch size, expected time, and interface ergonomics;
- provenance, guideline version, annotator confidence, and reason codes;
- stopping criteria based on marginal performance per unit cost.

Query selection changes the annotation distribution. Uncertain examples are often genuinely harder, so raw annotator agreement can fall even while the labeling program becomes more useful. Training data may also cease to be an IID sample. Evaluation must remain on a representative, independently sampled, clean set, and propensity weighting may be needed when estimating population quantities from adaptively selected labels.


### **Weak Supervision**

Weak supervision replaces some manual instance labels with cheaper, noisier sources. Sources may include domain heuristics, pattern matches, external databases, distant supervision, crowd workers, legacy systems, or model predictions. The goal is not to declare these labels correct. It is to model their coverage, reliability, conflict, and dependence, then use the aggregated signal to train a model that operates on the original inputs.

#### **Heuristics, Distant Supervision, and Label Models**

A **labeling function** $\lambda_j(x)$ returns a class label or abstains. For $m$ functions and $n$ observations, the label matrix is

$$
\Lambda_{ij}\in\{-1,0,\ldots,K-1\},
$$

where $-1$ denotes abstention. Useful diagnostics include:

$$
\text{coverage}_j=\frac{1}{n}\sum_i\mathbf 1(\Lambda_{ij}\ne-1),
\qquad
\text{conflict}_{jk}=\frac{\sum_i\mathbf 1(\Lambda_{ij}\ne-1,\Lambda_{ik}\ne-1,\Lambda_{ij}\ne\Lambda_{ik})}
{\sum_i\mathbf 1(\Lambda_{ij}\ne-1,\Lambda_{ik}\ne-1)}.
$$

Majority vote treats every source as equally accurate and independent. A **label model** instead estimates latent source accuracies and possibly correlations, then produces probabilistic targets $p(y_i\mid\Lambda_i)$. The end model learns from raw features $x_i$, so it can generalize beyond the exact rules at inference time.

<div class="diagram-scroll">

![A weak-supervision pipeline from labeling functions to an end model.](assets/weak-supervision-label-model.svg){fig-alt="Heuristics, patterns, database matches, and crowd votes form an abstaining vote matrix; a label model estimates source quality and produces probabilistic targets used by an end model."}

</div>

Identifiability is the central difficulty. If three keyword rules are minor rewrites of one another, their agreement is not three independent pieces of evidence. An unsupervised label model may overestimate them unless dependence is modeled. A small clean development set remains valuable for writing rules, detecting polarity mistakes, validating source assumptions, calibrating probabilistic labels, and evaluating the end model.

<details>
<summary><strong>Python: build and audit simple labeling functions</strong></summary>

```python
import numpy as np

ABSTAIN, NEGATIVE, POSITIVE = -1, 0, 1
texts = [
    "claim your free cash prize now",
    "project meeting moved to Friday",
    "urgent winner click http offer",
    "thanks for the helpful review",
    "limited discount available today",
    "team notes and sprint plan",
    "verify your account at http link",
    "the product failed after one day",
    "excellent service and fast delivery",
    "free workshop for the research team",
]

def lf_promotional_terms(text):
    terms = {"free", "cash", "prize", "winner", "discount", "offer"}
    return POSITIVE if terms.intersection(text.lower().split()) else ABSTAIN

def lf_link_and_urgency(text):
    lowered = text.lower()
    return POSITIVE if "http" in lowered and ("urgent" in lowered or "verify" in lowered) else ABSTAIN

def lf_work_context(text):
    terms = {"project", "meeting", "team", "sprint", "research"}
    return NEGATIVE if terms.intersection(text.lower().split()) else ABSTAIN

functions = [lf_promotional_terms, lf_link_and_urgency, lf_work_context]
Lambda = np.array([[function(text) for function in functions] for text in texts])

coverage = (Lambda != ABSTAIN).mean(axis=0)
overlap = ((Lambda != ABSTAIN).sum(axis=1) > 1).mean()
conflict = np.array([
    len(set(row[row != ABSTAIN])) > 1 for row in Lambda
]).mean()

# A transparent weighted vote is a baseline, not a learned label model.
weights = np.array([0.8, 0.9, 0.75])
signed_votes = np.where(Lambda == ABSTAIN, 0, np.where(Lambda == POSITIVE, 1, -1))
score = signed_votes @ weights
weak_probability = 1 / (1 + np.exp(-score))

print("vote matrix (rows=documents, columns=functions):\n", Lambda)
print("coverage:", np.round(coverage, 2).tolist())
print("overlap rate:", round(float(overlap), 2))
print("conflict rate:", round(float(conflict), 2))
print("weak positive probabilities:", np.round(weak_probability, 2).tolist())
```

</details>

The English examples remain unchanged because the keyword rules are part of the experiment. Before deployment, inspect coverage and conflict by subgroup, version rules like code, test them against labeled fixtures, and prevent train-test leakage from dictionaries or databases created with future information.

#### **Positive-Unlabeled Learning**

Positive-unlabeled (PU) learning observes a selected set of positive examples and an unlabeled set containing both positive and negative cases. Treating every unlabeled example as negative creates systematic false negatives. Let $y\in\{0,1\}$ be the true target and $s=1$ indicate that an example was selected into the labeled-positive set. Under **selected completely at random** (SCAR),

$$
P(s=1\mid y=1,x)=c,
\qquad
P(s=1\mid y=0,x)=0.
$$

Then a classifier for selection learns

$$
g(x)=P(s=1\mid x)=cP(y=1\mid x),
\qquad
P(y=1\mid x)=\frac{g(x)}{c}.
$$

<div class="diagram-scroll">

![Positive-unlabeled learning under the SCAR assumption.](assets/pu-learning-mixture.svg){fig-alt="Selected positives and an unlabeled mixture of hidden positives and negatives train a selection classifier; under SCAR the selection probability is divided by the positive selection rate."}

</div>

The constant $c$ can be estimated from held-out known positives under SCAR. If easy, recent, urban, high-value, or severe positives are more likely to be labeled, selection is **selected at random** conditional on features or is otherwise biased, and one constant is invalid. The selection mechanism must then be modeled explicitly or supported by additional labels.

An alternative risk formulation uses the positive class prior $\pi=P(y=1)$:

$$
R(f)=\pi\,\mathbb E_P[\ell(f(x),1)]
+\mathbb E_U[\ell(f(x),0)]
-\pi\,\mathbb E_P[\ell(f(x),0)].
$$

The last two terms recover the negative risk from the unlabeled mixture. Finite-sample estimates can become negative, motivating non-negative PU objectives that clip or otherwise control this component.

<details>
<summary><strong>Python: apply the Elkan-Noto probability correction under SCAR</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=5000, n_features=18, n_informative=9,
    weights=[0.78, 0.22], class_sep=1.1, random_state=153
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=153
)
rng = np.random.default_rng(153)

# Reveal each true training positive independently with probability c.
true_c = 0.35
s = np.zeros_like(y_train)
positive_index = np.flatnonzero(y_train == 1)
revealed = positive_index[rng.random(len(positive_index)) < true_c]
s[revealed] = 1

# Hold some revealed positives out when estimating c.
rng.shuffle(revealed)
c_validation = revealed[: max(20, len(revealed) // 5)]
fit_mask = np.ones(len(X_train), dtype=bool)
fit_mask[c_validation] = False

selection_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
).fit(X_train[fit_mask], s[fit_mask])

estimated_c = selection_model.predict_proba(X_train[c_validation])[:, 1].mean()
selection_probability = selection_model.predict_proba(X_test)[:, 1]
pu_probability = np.clip(selection_probability / estimated_c, 0, 1)

print("true c / estimated c:", round(true_c, 3), round(float(estimated_c), 3))
print("selection AUC:", round(roc_auc_score(y_test, selection_probability), 3))
print("raw selection Brier score:", round(brier_score_loss(y_test, selection_probability), 3))
print("PU-corrected Brier score:", round(brier_score_loss(y_test, pu_probability), 3))
```

</details>

The correction changes probability scale, not ranking, so AUC is unchanged while calibration can improve. Clipping is a practical numerical step, not a substitute for checking SCAR. Evaluate on a clean sample containing both positives and negatives.


### **Learning with Label Noise**

Label noise means the recorded target $\tilde y$ differs from the target $y$ the model should predict. It may arise from measurement failure, ambiguous guidelines, automated extraction, careless annotation, temporal drift, or class definitions that changed after data collection. Noise in $x$ and noise in $y$ are different problems; class imbalance is also not label noise.

#### **Noise Models and Transition Matrices**

Common noise models include:

| Noise model | Corruption mechanism | Example | Main danger |
|---|---|---|---|
| Symmetric | Any class flips at a common rate | Random label replacement | Simplified benchmark may be unrealistically easy |
| Class-conditional | Flip rate depends on the true class | One disease confused with a similar disease | Requires an estimable class transition matrix |
| Instance-dependent | Flip rate also depends on $x$ | Blurry images or sarcastic text are mislabeled more often | Hard to identify without clean anchors or extra assumptions |
| Open-set | Some observations do not belong to any target class | Foreign objects in a closed-set dataset | Relabeling among known classes cannot fix ontology mismatch |
| Structured / group-dependent | Corruption follows source, annotator, time, or subgroup | One site uses an obsolete guideline | Aggregate noise rates hide systematic harm |

For class-conditional noise, define

$$
T_{ab}=P(\tilde y=b\mid y=a),
\qquad
\sum_b T_{ab}=1.
$$

If $p(x)$ is a row vector of clean class probabilities, then

$$
P(\tilde y\mid x)=p(x)T.
$$

**Forward correction** passes predicted clean probabilities through $T$ before comparing with observed labels. **Backward correction** transforms losses using $T^{-1}$. Forward correction is often numerically safer, while both depend on a sufficiently accurate and identifiable transition matrix.

<div class="diagram-scroll">

![A label-noise channel and three defense families.](assets/noisy-label-defense-map.svg){fig-alt="A true label passes through a symmetric, class-conditional, or instance-dependent transition channel to an observed label, followed by loss correction, sample selection, or auditing and relabeling."}

</div>

<details>
<summary><strong>Python: simulate class-conditional noise and invert a known transition matrix</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_blobs(
    n_samples=3600,
    centers=[(-2.2, -1.2), (2.0, -0.7), (0.0, 2.2)],
    cluster_std=[1.35, 1.25, 1.30],
    random_state=154,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=154
)

# Rows are true classes; columns are observed classes.
T = np.array([
    [0.82, 0.14, 0.04],
    [0.08, 0.78, 0.14],
    [0.18, 0.05, 0.77],
])
rng = np.random.default_rng(154)
y_noisy = np.array([rng.choice(3, p=T[class_id]) for class_id in y_train])

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500),
).fit(X_train, y_noisy)

# The fitted probabilities approximate P(observed label | x).
noisy_probability = model.predict_proba(X_test)
clean_probability = noisy_probability @ np.linalg.inv(T)
clean_probability = np.clip(clean_probability, 0, None)
clean_probability /= clean_probability.sum(axis=1, keepdims=True)

raw_prediction = noisy_probability.argmax(axis=1)
corrected_prediction = clean_probability.argmax(axis=1)

print("realized training noise rate:", round(float(np.mean(y_noisy != y_train)), 3))
print("raw noisy-label accuracy:", round(accuracy_score(y_test, raw_prediction), 3))
print("transition-corrected accuracy:", round(accuracy_score(y_test, corrected_prediction), 3))
print("condition number of T:", round(float(np.linalg.cond(T)), 2))
```

</details>

This controlled example knows $T$. In practice, estimating $T$ may require trusted anchor examples, repeated labels, a clean subset, or strong identifiability assumptions. An ill-conditioned matrix magnifies probability error during inversion. A transition matrix also cannot represent instance-dependent corruption or a missing class.

#### **Robust Losses and Sample Reweighting**

Cross-entropy assigns a large penalty to confident disagreement with the observed label. High-capacity models can eventually memorize corrupted targets, so early training dynamics sometimes distinguish easier clean patterns from later noise. Robust approaches include:

- label smoothing, bootstrapping, or soft targets;
- losses with bounded influence, such as mean absolute error or generalized cross-entropy;
- early stopping and strong regularization;
- sample weights derived from out-of-fold label quality, agreement, or provenance;
- loss correction with an estimated transition model;
- filtering or relabeling suspicious observations.

Generalized cross-entropy for observed class $\tilde y$ can be written

$$
\mathcal L_q(p_{\tilde y})=\frac{1-p_{\tilde y}^{q}}{q},
\qquad 0<q\le1.
$$

As $q\to0$, it approaches cross-entropy $-\log p_{\tilde y}$; at $q=1$, it becomes $1-p_{\tilde y}$, an MAE-like bounded loss. Greater robustness may reduce the gradient available for genuinely hard clean examples. No loss universally separates difficult signal from corruption.

Sample reweighting has the same risk. If the current model regards minority, novel, or boundary cases as low quality, down-weighting by model confidence can erase exactly the examples the system needs. Quality estimates should be out-of-fold, stratified by source and subgroup, and audited manually.

#### **Co-Teaching and Confident Learning**

**Co-teaching** trains two models with different initialization or data order. In each mini-batch, each model selects a small-loss subset and gives that subset to its peer for updating. Cross-updating reduces immediate self-confirmation, while a schedule gradually changes the retained fraction according to an assumed or estimated noise rate.

```text
for each mini-batch B:
    losses_A = model_A.loss_per_example(B)
    losses_B = model_B.loss_per_example(B)
    clean_for_B = smallest_loss_examples(losses_A, remember_rate)
    clean_for_A = smallest_loss_examples(losses_B, remember_rate)
    update(model_A, clean_for_A)
    update(model_B, clean_for_B)
```

The small-loss heuristic relies on early-learning behavior and can fail when hard clean examples systematically have larger loss. Two models can also become correlated and agree on the same error.

**Confident learning** focuses on label quality rather than prediction confidence alone. It uses out-of-sample predicted probabilities, class-specific thresholds, and counts of confident joint events to estimate label issues and rank suspicious examples. The important operational rule is that label-quality scores for training examples should come from models that did not train on those examples.

<details>
<summary><strong>Python: rank likely label issues using out-of-fold probabilities</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y_clean = make_classification(
    n_samples=2500, n_features=14, n_informative=8,
    class_sep=1.2, random_state=155
)
rng = np.random.default_rng(155)
y_observed = y_clean.copy()

# Corrupt 18% of class 1 and 6% of class 0.
flip_1 = rng.choice(np.flatnonzero(y_clean == 1), int(0.18 * np.sum(y_clean == 1)), replace=False)
flip_0 = rng.choice(np.flatnonzero(y_clean == 0), int(0.06 * np.sum(y_clean == 0)), replace=False)
flipped = np.concatenate([flip_1, flip_0])
y_observed[flipped] = 1 - y_observed[flipped]

estimator = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=155)
oof_probability = cross_val_predict(
    estimator, X, y_observed, cv=cv, method="predict_proba", n_jobs=1
)

# Probability assigned to the provided label is a simple label-quality score.
label_quality = oof_probability[np.arange(len(y_observed)), y_observed]
flagged = np.argsort(label_quality)[: len(flipped)]

precision_at_known_noise_count = np.mean(np.isin(flagged, flipped))
print("realized noise rate:", round(len(flipped) / len(y_observed), 3))
print("precision among lowest-quality labels:", round(float(precision_at_known_noise_count), 3))
print("ten most suspicious indexes:", flagged[:10].tolist())
```

</details>

The true number of flips is known only in this simulation. In real data, estimate issue counts cautiously and send ranked cases to review rather than automatically replacing every label with the model prediction. A confident disagreement may expose a label error, model blind spot, missing class, or ambiguous ontology.


### **Class Imbalance and Rare Events**

Class imbalance describes unequal class prevalence, not incorrect labels. In fraud detection, safety monitoring, diagnosis, or fault prediction, the positive class may be rare but operationally important. A classifier that predicts only the majority class can have high accuracy while providing no value.

The problem has three separate layers:

1. **Estimation:** the learner sees few minority examples and may fit them poorly.
2. **Evaluation:** metrics such as accuracy or ROC AUC may hide poor positive precision at realistic prevalence.
3. **Decision:** false negatives and false positives have different consequences, and review capacity may be limited.

#### **Resampling, Cost Sensitivity, and Thresholding**

**Random oversampling** duplicates minority examples, increasing their gradient frequency but not their information. **Random undersampling** reduces majority dominance and computation but may discard useful boundary cases. Synthetic methods such as SMOTE interpolate minority neighbours; they can help tabular problems but may create invalid observations, bridge distinct minority modes, or leak when applied before cross-validation. Every resampling operation belongs inside the training fold.

**Cost-sensitive learning** changes the training objective through class or sample weights. For binary cross-entropy,

$$
\mathcal L
=-w_1y\log p-w_0(1-y)\log(1-p).
$$

Weights change the fitted decision surface and can alter probability calibration. **Thresholding** instead changes the action after estimating a score. If $p=P(y=1\mid x)$ is calibrated, a false positive costs $C_{FP}$, and a false negative costs $C_{FN}$, predict positive when

$$
C_{FP}(1-p)\le C_{FN}p
\quad\Longleftrightarrow\quad
p\ge\frac{C_{FP}}{C_{FP}+C_{FN}}.
$$

Real policies may add investigation cost, capacity constraints, abstention, or class-dependent benefit, so the threshold should be validated against the actual utility function.

<div class="diagram-scroll">

![Rare-event score distributions and threshold trade-offs.](assets/imbalance-thresholding.svg){fig-alt="Overlapping score distributions for majority and rare-event classes are crossed by low and high thresholds, showing recall, precision, missed-event, and review-workload trade-offs."}

</div>

Precision-recall curves are especially informative under severe imbalance because precision directly reflects false-positive burden at the evaluated prevalence. Average precision summarizes ranking but does not select an operating point. Also report the confusion matrix, recall at a workload or precision constraint, calibration, subgroup performance, and uncertainty intervals.

<details>
<summary><strong>Python: separate model weighting from cost-based threshold selection</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=12000, n_features=20, n_informative=10,
    weights=[0.98, 0.02], class_sep=1.0, flip_y=0.005, random_state=156
)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=156
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=156
)

def fit_model(class_weight=None):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=class_weight, max_iter=2500),
    ).fit(X_train, y_train)

plain = fit_model()
balanced = fit_model("balanced")

# Choose a threshold on validation data using explicit business costs.
valid_probability = plain.predict_proba(X_valid)[:, 1]
false_positive_cost, false_negative_cost = 1.0, 25.0
threshold_grid = np.linspace(0.001, 0.999, 999)
validation_cost = []
for threshold in threshold_grid:
    prediction = valid_probability >= threshold
    fp = np.sum((prediction == 1) & (y_valid == 0))
    fn = np.sum((prediction == 0) & (y_valid == 1))
    validation_cost.append(false_positive_cost * fp + false_negative_cost * fn)
selected_threshold = threshold_grid[int(np.argmin(validation_cost))]

rows = []
for name, model, threshold in [
    ("plain @ 0.5", plain, 0.5),
    ("balanced @ 0.5", balanced, 0.5),
    ("plain @ cost threshold", plain, selected_threshold),
]:
    probability = model.predict_proba(X_test)[:, 1]
    prediction = probability >= threshold
    rows.append({
        "policy": name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision": average_precision_score(y_test, probability),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction),
        "alerts": int(prediction.sum()),
    })

print(pd.DataFrame(rows).round(3).to_string(index=False))
```

</details>

Class weighting and thresholding answer different questions. The weighted model changes parameter estimation, while the validation-selected threshold changes the action policy for one fitted probability model. Recalibrate after weighting or resampling, and retune thresholds when prevalence, costs, or capacity changes.


### **Evaluation under Imperfect Supervision**

Evaluation is hardest precisely when training labels are unreliable. A model cannot be validated by comparing it only with the same heuristics, pseudo-labels, or noisy source that trained it. The minimum defensible design separates four roles:

- **imperfect training evidence:** clean seeds, unlabeled data, weak votes, or noisy labels;
- **clean validation data:** model selection, calibration, confidence thresholds, label-model design, and stopping;
- **a frozen clean test set:** final task performance, subgroup analysis, and uncertainty;
- **an annotation ledger:** number of labels, expert minutes, adjudications, source versions, and quality-control cost.

<div class="diagram-scroll">

![An evaluation architecture for imperfect supervision.](assets/imperfect-supervision-evaluation.svg){fig-alt="Imperfect training evidence flows through training and tuning, then into a clean validation set and a frozen clean test set; a warning prohibits pseudo-labels and weak votes from becoming test truth."}

</div>

The protocol should match the learning setting:

| Setting | Essential baselines | Horizontal axis | Quality checks |
|---|---|---|---|
| Semi-supervised | Same-model label-only baseline | Number of clean labels plus unlabeled pool size | Labeled-set seeds, pseudo-label precision, calibration |
| Self-supervised | Random initialization and supervised pretraining | Unlabeled compute and downstream labels | Linear probe, fine-tuning curve, augmentation ablation |
| Active learning | Random sampling and simple uncertainty | Annotation time or cost | Repeated pool orders, query diversity, annotator difficulty |
| Weak supervision | Best single source and majority vote | Rule-development plus audit cost | Coverage, conflict, dependence, source ablation |
| PU learning | Naive unlabeled-as-negative baseline | Labeled positives and selection assumptions | Class prior, SCAR/SAR sensitivity, calibration |
| Label noise | Ordinary ERM and clean-data upper bound | Noise rate and clean-audit budget | Noise type, transition identifiability, flagged-label precision |
| Rare events | Unweighted model and prevalence baseline | Alerts, review capacity, or cost | Precision-recall, calibration, subgroup recall |

**Budget curves, repeated seeds, and clean test data.**

A single final score hides the purpose of limited-supervision methods. Plot performance against the scarce resource: clean labels, expert hours, weak-source development time, or review volume. The supervised baseline must use the same labeled subset and model capacity. Repeat the experiment because results at small budgets are dominated by which examples happened to be labeled.

<details>
<summary><strong>Python: construct repeated clean-label budget curves</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

X, y = make_moons(n_samples=2200, noise=0.24, random_state=157)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=157
)
base = make_pipeline(StandardScaler(), LogisticRegression(C=2.0, max_iter=2000))

records = []
for budget in [10, 20, 40, 80, 160]:
    for seed in range(12):
        rng = np.random.default_rng(2000 + 100 * budget + seed)
        per_class = budget // 2
        labeled = np.concatenate([
            rng.choice(np.flatnonzero(y_development == class_id), per_class, replace=False)
            for class_id in [0, 1]
        ])
        y_semi = np.full_like(y_development, -1)
        y_semi[labeled] = y_development[labeled]

        supervised = clone(base).fit(X_development[labeled], y_development[labeled])
        semi = SelfTrainingClassifier(
            estimator=clone(base), threshold=0.90, max_iter=20
        ).fit(X_development, y_semi)

        for method, model in [("label only", supervised), ("self-training", semi)]:
            records.append({
                "clean_labels": budget,
                "seed": seed,
                "method": method,
                "test_accuracy": accuracy_score(y_test, model.predict(X_test)),
            })

summary = (
    pd.DataFrame(records)
    .groupby(["method", "clean_labels"])["test_accuracy"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)
print(summary)
```

</details>

Because every method is paired on the same labeled subset for each seed, uncertainty in the difference can be estimated more efficiently than treating runs as independent.

<details>
<summary><strong>Python: bootstrap a paired performance difference across label seeds</strong></summary>

```python
import numpy as np
import pandas as pd

# Reuse the records DataFrame from the preceding example.
wide = (
    pd.DataFrame(records)
    .query("clean_labels == 40")
    .pivot(index="seed", columns="method", values="test_accuracy")
)
paired_difference = wide["self-training"] - wide["label only"]

rng = np.random.default_rng(158)
bootstrap_mean = np.array([
    rng.choice(paired_difference.to_numpy(), size=len(paired_difference), replace=True).mean()
    for _ in range(5000)
])

print("mean paired difference:", round(float(paired_difference.mean()), 4))
print("95% bootstrap interval:", np.round(np.quantile(bootstrap_mean, [0.025, 0.975]), 4))
```

</details>

The resampling unit is the labeled-set seed, not an individual test prediction, because the experiment asks how acquisition variability affects method performance. For a fixed deployed test set, a separate paired bootstrap over test observations can quantify prediction-level uncertainty.

**Audit questions and method selection.**

Before accepting an improvement, ask:

1. Is test truth independent of pseudo-labels, weak sources, and the query policy?
2. Does the comparison use the same clean-label, compute, and model-selection budget?
3. Are confidence thresholds, transition matrices, class weights, and stopping rules selected without test access?
4. Does performance improve across seeds, subgroups, and realistic shifts, or only on average?
5. Are probabilities calibrated at deployment prevalence?
6. What fraction of weak or pseudo-labels is covered, conflicted, abstained, corrected, or manually audited?
7. Could the method suppress hard clean examples, rare classes, or annotator disagreement?
8. Is the gain large enough to justify annotation infrastructure and operational complexity?

| Situation | Prefer | Avoid relying on |
|---|---|---|
| Many unlabeled examples with trustworthy geometry | Semi-supervised learning | Confidence without calibration |
| Large raw corpus and reusable representations | Self-supervised pretraining | Pretext loss as downstream evidence |
| Experts are available but expensive | Active learning | Query count as a proxy for all cost |
| Domain rules or external sources are abundant | Weak supervision | Majority vote over correlated rules |
| Only selected positives are known | PU learning | Treating unlabeled examples as negatives |
| Existing labels have identifiable corruption | Transition-aware or robust learning | Synthetic symmetric noise as the only test |
| A correct rare class drives asymmetric decisions | Cost-sensitive training and thresholding | Accuracy or a universal threshold |

The strongest practical strategy is often hybrid: self-supervised pretraining reduces representation cost, active learning builds a small clean set, weak sources expand coverage, robust training limits noisy-label damage, and a pristine evaluation set keeps every stage honest. Complexity should be added only after a simple supervised baseline and a label-quality audit reveal the actual bottleneck.

Official and primary resources include the [scikit-learn semi-supervised learning guide](https://scikit-learn.org/stable/modules/semi_supervised.html), the [SimCLR paper](https://research.google/pubs/a-simple-framework-for-contrastive-learning-of-visual-representations/), the [active learning literature survey](https://research.cs.wisc.edu/techreports/2009/TR1648.pdf), the [Snorkel weak-supervision paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC5951191/), the [positive-unlabeled learning paper](https://doi.org/10.1145/1401890.1401920), the [Co-teaching paper](https://papers.nips.cc/paper/2018/hash/a19744e268754fb0148b017647355b7b-Abstract.html), the [confident learning paper](https://research.google/pubs/confident-learning-estimating-uncertainty-in-dataset-labels/), and the [scikit-learn precision-recall example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html).
